# Open-Loop Rollout Diagnostic

For a Dreamer v1 world model, this notebook tests whether the model can predict
future observations using only the prior — i.e., without seeing observations
during the rollout. This is the diagnostic that matters for whether imagination-based
policy training will work, since the actor's training trajectory is a prior rollout.

**What we check:**
1. Closed-loop reconstruction is sharp (sanity).
2. Open-loop predictions track reality across `horizon` steps.
3. Open-loop error beats a zero-dynamics baseline ("predict no change").
4. Per-dim error highlights which observation components the model handles well.
5. Reward prediction is accurate over the horizon (what the actor actually relies on).

Heavy lifting (episode loading, rollout, metrics) lives in `chuck_dreamer.eval`.
The notebook itself does only plotting and per-dim analysis.

In [ ]:
# Parameters — papermill injects overrides after this cell.
checkpoint_path = "checkpoints/default/latest.safetensors"
data_path       = "data/eval/"
data_format     = "rerun"
num_episodes    = 20
burn_in         = 5
horizon         = 15
seed            = 0

## Setup

In [ ]:
import json

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np

from chuck_dreamer.eval import (
  load_checkpoint, load_eval_episodes, run_split_rollout,
  recon_l2, reward_abs_error, zero_dynamics_prediction, stack_curves,
)

np.random.seed(seed)
mx.random.seed(seed)

print(f"checkpoint:    {checkpoint_path}")
print(f"data:          {data_path} ({data_format})")
print(f"num_episodes:  {num_episodes}")
print(f"burn_in:       {burn_in}")
print(f"horizon:       {horizon}")

## Load model and evaluation episodes

In [ ]:
ckpt = load_checkpoint(checkpoint_path)
obs_mode = ckpt.env.obs_mode
print(f"obs_mode={obs_mode}  act_mode={ckpt.env.act_mode}")
print(f"RSSM dims: stoch={ckpt.model.rssm.stoch_dim}, deter={ckpt.model.rssm.deter_dim}")

sources = [{"path": data_path, "format": data_format, "num_episodes": num_episodes}]
T_window = burn_in + horizon
results = []
for episode in load_eval_episodes(ckpt.config, sources=sources,
                                  min_len=T_window, max_episodes=num_episodes):
  out = run_split_rollout(
    ckpt.model, episode,
    burn_in=burn_in, horizon=horizon, obs_mode=obs_mode,
    decode=True, predict_reward=True,
  )
  if out is not None:
    results.append(out)
if not results:
  raise RuntimeError(f"No usable eval episodes (need {T_window}+ actions).")
print(f"Computed rollouts for {len(results)} episodes")

## Diagnostic 1 — Open-loop error vs horizon

Per-step L2 error between predicted and true observation, averaged over episodes.
The vertical line marks the burn-in boundary. The **zero-dynamics baseline** is
"predict that the obs stays equal to the last burn-in obs forever" — your model
should beat it after a few open-loop steps.

In [ ]:
prior_curves = [recon_l2(r.recon_prior,     r.target_obs) for r in results]
post_curves  = [recon_l2(r.recon_posterior, r.target_obs) for r in results]
if burn_in > 0:
  zero_curves = [
    recon_l2(zero_dynamics_prediction(r.full_target_obs, burn_in - 1), r.full_target_obs)
    for r in results
  ]
else:
  zero_curves = []

prior_arr = stack_curves(prior_curves)
post_arr  = stack_curves(post_curves)
zero_arr  = stack_curves(zero_curves) if zero_curves else np.empty((0, 0))

mean_prior = np.nanmean(prior_arr, axis=0)
std_prior  = np.nanstd(prior_arr,  axis=0)
mean_post  = np.nanmean(post_arr,  axis=0)
mean_zero  = np.nanmean(zero_arr,  axis=0) if zero_arr.size else None

fig, ax = plt.subplots(figsize=(9, 4))
xs_open = np.arange(burn_in, burn_in + mean_prior.shape[0])
ax.plot(xs_open, mean_prior, label="prior (open-loop)", color="C0", linewidth=2)
ax.fill_between(xs_open, mean_prior - std_prior, mean_prior + std_prior, alpha=0.2, color="C0")
ax.plot(xs_open, mean_post,  label="posterior (teacher-forced)", color="C1", linewidth=2)
if mean_zero is not None:
  ax.plot(np.arange(mean_zero.shape[0]), mean_zero,
          label="zero-dynamics baseline", color="C3", linestyle="--", linewidth=1.5)
ax.axvline(burn_in - 0.5, color="k", linestyle=":", alpha=0.5, label="burn-in")
ax.set_xlabel("step (action index)")
ax.set_ylabel(f"obs L2 error ({obs_mode})")
ax.set_title("Rollout error vs horizon")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Diagnostic 2 — Per-dim / per-component open-loop error

For state-mode obs we bar-chart the mean absolute prior-tail error per dim,
colored by component (ee_pos, ee_quat, object_xy, joint_qpos). For image
modes we render real vs predicted frames side-by-side at a few open-loop
timesteps.

In [ ]:
def _state_layout(obs_dim):
  n_joints = obs_dim - 9
  groups = {"ee_pos": list(range(0, 3)), "ee_quat": list(range(3, 7)),
            "object_xy": list(range(7, 9)),
            "joint_qpos": list(range(9, obs_dim))}
  labels = (["ee_x", "ee_y", "ee_z"]
            + ["ee_qw", "ee_qx", "ee_qy", "ee_qz"]
            + ["obj_x", "obj_y"]
            + [f"q_{i}" for i in range(n_joints)])
  return groups, labels

def _proprio_layout(proprio_dim):
  n_joints = proprio_dim - 7
  groups = {"ee_pos": list(range(0, 3)), "ee_quat": list(range(3, 7)),
            "joint_qpos": list(range(7, proprio_dim))}
  labels = (["ee_x", "ee_y", "ee_z"]
            + ["ee_qw", "ee_qx", "ee_qy", "ee_qz"]
            + [f"q_{i}" for i in range(n_joints)])
  return groups, labels

def _mean_abs_err_per_dim(get_pred, get_true):
  diffs = [np.abs(get_pred(r) - get_true(r)) for r in results]
  return np.concatenate(diffs, axis=0).mean(axis=0)

def _plot_per_dim(per_dim, groups, labels, title):
  colors = []
  for i in range(len(per_dim)):
    for g_idx, (_, dims) in enumerate(groups.items()):
      if i in dims:
        colors.append(f"C{g_idx}"); break
    else:
      colors.append("gray")
  fig, ax = plt.subplots(figsize=(10, 4))
  ax.bar(np.arange(len(per_dim)), per_dim, color=colors)
  ax.set_xticks(np.arange(len(per_dim)))
  ax.set_xticklabels(labels, rotation=45, ha="right")
  ax.set_ylabel("mean |error| in open-loop region")
  ax.set_title(title)
  for g_idx, name in enumerate(groups):
    ax.bar([], [], color=f"C{g_idx}", label=name)
  ax.legend(); ax.grid(alpha=0.3, axis="y")
  plt.tight_layout(); plt.show()

def _plot_image_strip(get_pred, get_true, title):
  r = results[0]
  pred = get_pred(r); true = get_true(r)
  T = pred.shape[0]
  ts = [t for t in (0, 2, 5, 10, T - 1) if 0 <= t < T]
  fig, axes = plt.subplots(2, len(ts), figsize=(3 * len(ts), 6))
  axes = np.atleast_2d(axes)
  for col, t in enumerate(ts):
    axes[0, col].imshow(np.clip((true[t] + 0.5) * 255, 0, 255).astype(np.uint8))
    axes[0, col].set_title(f"real t={burn_in + t}"); axes[0, col].axis("off")
    axes[1, col].imshow(np.clip((pred[t] + 0.5) * 255, 0, 255).astype(np.uint8))
    axes[1, col].set_title(f"pred t={burn_in + t}"); axes[1, col].axis("off")
  fig.suptitle(title); plt.tight_layout(); plt.show()

if obs_mode == "state":
  obs_dim = results[0].target_obs.shape[-1]
  groups, labels = _state_layout(obs_dim)
  per_dim = _mean_abs_err_per_dim(lambda r: r.recon_prior, lambda r: r.target_obs)
  _plot_per_dim(per_dim, groups, labels,
                f"Per-dim prior-tail error, steps {burn_in}–{burn_in + horizon - 1}")
elif obs_mode == "image":
  _plot_image_strip(lambda r: r.recon_prior, lambda r: r.target_obs,
                    "Episode 0 — real (top) vs prior-predicted (bottom)")
elif obs_mode == "image_proprio":
  _plot_image_strip(lambda r: r.recon_prior["image"],
                    lambda r: r.target_obs["image"],
                    "Episode 0 — real (top) vs prior-predicted (bottom)")
  proprio_dim = results[0].target_obs["proprio"].shape[-1]
  groups, labels = _proprio_layout(proprio_dim)
  per_dim = _mean_abs_err_per_dim(lambda r: r.recon_prior["proprio"],
                                  lambda r: r.target_obs["proprio"])
  _plot_per_dim(per_dim, groups, labels,
                f"Per-dim proprio prior-tail error, steps {burn_in}–{burn_in + horizon - 1}")

## Diagnostic 3 — Reward prediction error

The actor relies on the reward predictor during imagination. Reward MAE under
the prior tail is the metric most directly predictive of whether the actor will
train successfully.

In [ ]:
rew_prior = stack_curves([reward_abs_error(r.reward_prior,     r.reward) for r in results])
rew_post  = stack_curves([reward_abs_error(r.reward_posterior, r.reward) for r in results])

all_rewards = np.concatenate([r.reward for r in results])
mean_reward = float(all_rewards.mean()) if all_rewards.size else 0.0
rew_base = stack_curves([np.abs(mean_reward - r.reward) for r in results])

mean_prior_rew = np.nanmean(rew_prior, axis=0)
mean_post_rew  = np.nanmean(rew_post,  axis=0)
mean_base_rew  = np.nanmean(rew_base,  axis=0)
std_prior_rew  = np.nanstd(rew_prior,  axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
xs = np.arange(burn_in, burn_in + mean_prior_rew.shape[0])
ax.plot(xs, mean_prior_rew, label="prior reward MAE",     color="C2", linewidth=2)
ax.fill_between(xs, mean_prior_rew - std_prior_rew, mean_prior_rew + std_prior_rew,
                alpha=0.2, color="C2")
ax.plot(xs, mean_post_rew, label="posterior reward MAE", color="C1", linewidth=2)
ax.plot(xs, mean_base_rew, label="mean-predictor baseline",
        color="C3", linestyle="--", linewidth=1.5)
ax.set_xlabel("step"); ax.set_ylabel("|predicted - true| reward")
ax.set_title("Reward prediction error vs horizon")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

Track these across checkpoints — the headline numbers for world-model quality.

In [ ]:
def _at(arr, idx):
  return float(arr[idx]) if arr is not None and 0 <= idx < arr.shape[0] else None

summary = {
  "checkpoint":             str(checkpoint_path),
  "obs_mode":               obs_mode,
  "act_mode":               ckpt.env.act_mode,
  "num_episodes":           len(results),
  "burn_in":                burn_in,
  "horizon":                horizon,
  "prior_obs_err_at_h1":    _at(mean_prior, 0),
  "prior_obs_err_at_h5":    _at(mean_prior, 4),
  "prior_obs_err_at_h15":   _at(mean_prior, 14),
  "post_obs_err_at_h1":     _at(mean_post,  0),
  "post_obs_err_at_h5":     _at(mean_post,  4),
  "post_obs_err_at_h15":    _at(mean_post,  14),
  "zero_dyn_obs_err_at_h1": _at(mean_zero,  burn_in)     if mean_zero is not None else None,
  "zero_dyn_obs_err_at_h5": _at(mean_zero,  burn_in + 4) if mean_zero is not None else None,
  "prior_rew_err_at_h5":    _at(mean_prior_rew, 4),
  "prior_rew_err_at_h15":   _at(mean_prior_rew, 14),
  "rew_baseline_at_h5":     _at(mean_base_rew,  4),
}
print(json.dumps(summary, indent=2))